# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive workflow for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All dataset elements (record sets, fields, columns) are referenced using their unique `@id` notation, ensuring consistent and reproducible analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
meta = dataset.metadata
print(f"Dataset: {meta.name}\nDescription: {meta.description}\nPublished: {getattr(meta, 'datePublished', None)}")

## 2. Data Overview
Explore the available record sets, fields, and their `@id` attributes within the dataset.

In [ ]:
# List all available record sets and their fields
print("Available RecordSets and their fields:")
record_sets = list(meta.record_sets)
for rec in record_sets:
    print(f"- RecordSet name: {getattr(rec, 'name', '<no_name>')}")
    print(f"  @id: {rec['@id']}")
    print("  Fields:")
    for fld in getattr(rec, 'fields', []):
        print(f"    - {fld['name']} (@id: {fld['@id']}) @type: {fld.get('@type', None)}")
    print('')

### Preview Records
You may view the first few records from a chosen record set, referencing it by its `@id`.

In [ ]:
# Pick the primary record set by @id (adjust based on printed output above)
record_set_id = None
for rec in record_sets:
    if ('@id' in rec) and ('main' in rec['@id'].lower() or True):  # Pick the first available
        record_set_id = rec['@id']
        break
if not record_set_id:
    raise RuntimeError('No record sets found.')

print(f"First 3 records from RecordSet @id {record_set_id}:")
for i, rec in enumerate(dataset.records(record_set=record_set_id)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data for all record sets into pandas DataFrames for analysis, using each record set's `@id`.

In [ ]:
# Load all record sets into DataFrames by @id
all_record_set_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for rec_id in all_record_set_ids:
    recs = list(dataset.records(record_set=rec_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rec_id}, shape: {df.shape}")
    else:
        print(f"No records found for RecordSet @id: {rec_id}")

# Display columns in the main record set
main_rs_id = record_set_id  # reusing detected record set above
print(f"Columns in main record set (@id {main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering by numeric value, normalization, and grouping. Choose numeric and group fields by their `@id` attributes (refer to overview above, adjust as needed in code).

In [ ]:
# === User must pick suitable field @ids for numeric analysis (change below based on overview printouts) ===
main_df = dataframes[main_rs_id]

# Try to auto-detect a numeric column
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break
if not numeric_field:
    # failing that, try to find a well-known numeric field name
    candidates = ['abundance', 'MW', 'molecular_weight', 'count', 'peptide_count', 'value', 'coverage']
    for col in main_df.columns:
        if col.lower() in candidates:
            numeric_field = col
            main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
            break
if not numeric_field:
    raise RuntimeError('No numeric field found.')

print(f"Using numeric field for EDA: {numeric_field}")

# Filtering
threshold = main_df[numeric_field].mean() if main_df[numeric_field].mean() > 0 else 10
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a categorical field (choose one by @id, e.g. by protein type or accession)
group_field = None
for cand in ['accession', 'description', 'category', 'group', 'class', 'protein', 'name']:
    if cand in filtered_df.columns:
        group_field = cand
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No groupable categorical field found in DataFrame for grouping.")

## 5. Visualization
Visualize the filtered and normalized numeric field using common plots (histogram, boxplot).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Histogram of {numeric_field} (filtered > {threshold:.2f})")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(6,3))
sns.boxplot(x=filtered_df[numeric_field])
plt.title(f"Boxplot of {numeric_field} (filtered > {threshold:.2f})")
plt.xlabel(numeric_field)
plt.show()

# If grouping worked, show a barplot
if group_field:
    top_groups = grouped_df.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_groups.index, y=top_groups.values)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- Using the Croissant FAIR<sup>2</sup> schema and `mlcroissant`, we explored the main table in the mass spectrometry dataset.
- Fields such as protein accession, description, and key numeric metrics (e.g., abundance, MW, coverage percent) can be referenced directly using their `@id` (column name/label).
- We demonstrated filtering, normalization, grouping, and visualization for a selected numeric field, providing a starting point for further proteomic analysis and research.

Feel free to adapt the field and group variable selections based on your analysis needs and the specific `@id` values available in this dataset.